# 🎬 AI Multi-Speaker Video Dubbing Studio (Cloud Edition)
### 100% Free • Runs in Google Cloud (No installation required on your PC)

This notebook runs the complete AI Video Dubbing app on Google's free servers with **Free NVIDIA GPU acceleration**.

### ⚡ Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** (or CPU).
2. Click **Runtime** > **Run all** (or press `Ctrl + F9`).
3. Click the generated **Public URL** at the bottom of the final cell to open your Dubbing Studio in your browser!

In [ ]:
# Step 1: Install FFmpeg & Python Dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q fastapi uvicorn[standard] python-multipart aiofiles faster-whisper deep-translator edge-tts pyannote.audio soundfile numpy scipy scikit-learn librosa pyngrok

In [ ]:
# Step 2: Write voices.py
voices_code = '''
LANGUAGE_VOICES = {
    "hi": {"name": "Hindi (हिंदी)", "flag": "🇮🇳", "voices": [
        {"id": "hi-IN-MadhurNeural", "gender": "Male", "label": "Madhur (Male)"},
        {"id": "hi-IN-SwaraNeural", "gender": "Female", "label": "Swara (Female)"}
    ]},
    "ur": {"name": "Urdu (اردو)", "flag": "🇵🇰", "voices": [
        {"id": "ur-PK-AsadNeural", "gender": "Male", "label": "Asad (Male)"},
        {"id": "ur-PK-UzmaNeural", "gender": "Female", "label": "Uzma (Female)"}
    ]},
    "en": {"name": "English (US)", "flag": "🇺🇸", "voices": [
        {"id": "en-US-GuyNeural", "gender": "Male", "label": "Guy (Male)"},
        {"id": "en-US-JennyNeural", "gender": "Female", "label": "Jenny (Female)"},
        {"id": "en-US-ChristopherNeural", "gender": "Male", "label": "Christopher (Male)"},
        {"id": "en-US-AriaNeural", "gender": "Female", "label": "Aria (Female)"}
    ]},
    "es": {"name": "Spanish (Español)", "flag": "🇪🇸", "voices": [
        {"id": "es-ES-AlvaroNeural", "gender": "Male", "label": "Alvaro (Male)"},
        {"id": "es-ES-ElviraNeural", "gender": "Female", "label": "Elvira (Female)"}
    ]}
}
def get_voice_for_speaker(language, speaker_index):
    lang_info = LANGUAGE_VOICES.get(language, LANGUAGE_VOICES["en"])
    voices = lang_info["voices"]
    return voices[speaker_index % len(voices)]

def get_supported_languages():
    return [{"code": c, "name": d["name"], "flag": d["flag"]} for c, d in LANGUAGE_VOICES.items()]
'''
with open('voices.py', 'w', encoding='utf-8') as f:
    f.write(voices_code)
print('voices.py ready!')

In [ ]:
# Step 3: Launch Dubbing App with a Free Public Cloud Tunnel
import subprocess
import threading
import time
from google.colab import output

# Start localtunnel or pyngrok to expose port 8000
!npm install -g localtunnel > /dev/null 2>&1

# Get external IP for localtunnel verification
import urllib.request
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"\n🔑 LocalTunnel Password (if asked): {public_ip}\n")

# Start FastAPI server in background
def run_server():
    subprocess.run(["python", "app.py"])

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(3)

print("===========================================================")
print("🚀 Your App is starting! Click the public URL below:")
print("===========================================================")
!npx localtunnel --port 8000